# NYC Taxi – Raw Data Access Validation

## Objective
Validate connectivity between Azure Databricks and the NYC Taxi raw data stored in ADLS Gen2.

## Source
- Dataset: NYC TLC Yellow Taxi
- Period: January 2025
- Format: Parquet
- Storage: Azure Data Lake Storage Gen2
- Access method: Unity Catalog external location

## Expected Outcome
Confirm that Databricks can access and read the raw source data before implementing Bronze ingestion.

In [0]:
# Validate Spark session
print(f"Spark version: {spark.version}")

In [0]:
# Verify access to the project catalog and schemas

display(
    spark.sql("""
        SHOW SCHEMAS IN nyc_taxi
    """)
)

In [0]:
# Discover available Yellow Taxi data for 2025

year_path = (
    "abfss://raw@nyctaxidata2026abc.dfs.core.windows.net/"
    "yellow_taxi/year=2025/"
)

year_entries = dbutils.fs.ls(year_path)

display(year_entries)

In [0]:
# Discover Parquet files across the available 2025 month folders

parquet_files = []

for month_entry in year_entries:
    if month_entry.isDir():
        month_files = dbutils.fs.ls(month_entry.path)

        for file in month_files:
            if file.name.endswith(".parquet"):
                parquet_files.append(file)

display(parquet_files)

In [0]:
# Read the discovered Parquet file

source_path = parquet_files[0].path

df = spark.read.parquet(source_path)

# Inspect the source schema
df.printSchema()

In [0]:
# Inspect sample source records
display(df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Add ingestion metadata
bronze_df = (
    df
    .withColumn("source_file_path", input_file_name())
    .withColumn("ingestion_timestamp", current_timestamp())
)

# Write the initial dataset to the Bronze Delta table
target_table = "nyc_taxi.bronze.yellow_taxi"

(
    bronze_df.write
    .format("delta")
    .saveAsTable(target_table)
)

In [0]:
# Validate the Bronze Delta table

bronze_df = spark.table("nyc_taxi.bronze.yellow_taxi")

print("Bronze table schema:")
bronze_df.printSchema()

print("Sample Bronze records:")
display(bronze_df.limit(10))

In [0]:
# Compare source and Bronze record counts

source_count = df.count()
bronze_count = bronze_df.count()

print(f"Source records: {source_count:,}")
print(f"Bronze records: {bronze_count:,}")
print(f"Records match: {source_count == bronze_count}")

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Check for null values in Bronze metadata columns
metadata_check = bronze_df.agg(
    spark_sum(col("source_file_path").isNull().cast("int"))
        .alias("null_source_file_path"),

    spark_sum(col("ingestion_timestamp").isNull().cast("int"))
        .alias("null_ingestion_timestamp")
)

display(metadata_check)